# Waste Classification - Local GPU Training
Optimized for NVIDIA GeForce GTX 1650 Ti (4.29GB VRAM)

In [1]:
# Setup and Configurations
import os
import torch
import yaml
from PIL import Image
from pathlib import Path
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, ConcatDataset
from torchvision import transforms, models
import numpy as np
from tqdm.notebook import tqdm
import time
from datetime import datetime
import logging
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import gc

# GTX 1650 Ti Optimizations
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.enabled = True

# Constants
BATCH_SIZE = 48  # Optimized for 4.29GB VRAM
IMAGE_SIZE = 224
NUM_CLASSES = 4
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Local Paths
YOLO_PATH = r'C:\Users\divya\OneDrive\Desktop\ECS\GARBAGE_YOLO'
FOLDER_PATH = r'C:\Users\divya\OneDrive\Desktop\ECS\garbage_classification'

# Category mappings
YOLO_CATEGORY_MAP = {
    'BIODEGRADABLE': 0,
    'CARDBOARD': 0,
    'PAPER': 0,
    'GLASS': 3,
    'METAL': 2,
    'PLASTIC': 1
}

FOLDER_CATEGORY_MAP = {
    'battery': 2,
    'biological': 0,
    'brown-glass': 3,
    'cardboard': 0,
    'clothes': 0,
    'green-glass': 3,
    'metal': 2,
    'paper': 0,
    'plastic': 1,
    'shoes': 0,
    'trash': 0,
    'white-glass': 3
}

CLASS_NAMES = ['organic', 'plastic', 'metal', 'glass']

# Data transforms
data_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Memory tracking
def print_gpu_memory():
    if torch.cuda.is_available():
        print(f"GPU Memory Usage:")
        print(f"Allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")
        print(f"Cached: {torch.cuda.memory_reserved()/1e9:.2f} GB")

# Verify GPU setup
print(f"{'='*80}")
print(f"🚀 System Setup at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")
    print_gpu_memory()
print(f"{'='*80}")

🚀 System Setup at 2024-11-14 00:34:07
Device: cuda
GPU: NVIDIA GeForce GTX 1650 Ti
CUDA Version: 11.8
GPU Memory Usage:
Allocated: 0.00 GB
Cached: 0.00 GB


## Dataset Classes

In [2]:
class YOLOWasteDataset(Dataset):
    def __init__(self, image_dir, yaml_classes, transform=None):
        self.image_dir = Path(image_dir)
        self.label_dir = Path(str(image_dir).replace('images', 'labels'))
        self.transform = transform
        self.yaml_classes = yaml_classes
        
        print(f"\n📁 Loading YOLO Dataset from: {self.image_dir}")
        
        # Get all image files
        self.image_files = [f for f in os.listdir(self.image_dir) 
                           if f.endswith(('.jpg', '.jpeg', '.png'))]
        
        # Validate image-label pairs
        valid_files = []
        for img_file in tqdm(self.image_files, desc="Validating files"):
            label_file = Path(self.label_dir) / f"{Path(img_file).stem}.txt"
            if label_file.exists():
                valid_files.append(img_file)
                
        self.image_files = valid_files
        print(f"✓ Found {len(self.image_files):,} valid images")

    def __len__(self):
        return len(self.image_files)
    
    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(self.image_dir, img_name)
        label_path = os.path.join(self.label_dir, 
                                 img_name.rsplit('.', 1)[0] + '.txt')
        
        # Load image
        try:
            image = Image.open(img_path).convert('RGB')
            if self.transform:
                image = self.transform(image)
        except Exception as e:
            print(f"Error loading image {img_path}: {e}")
            # Return a blank image and default label if there's an error
            return torch.zeros((3, IMAGE_SIZE, IMAGE_SIZE)), 0
        
        # Load label
        try:
            with open(label_path, 'r') as f:
                class_idx = int(f.readline().split()[0])
                class_name = self.yaml_classes[class_idx]
                main_category = YOLO_CATEGORY_MAP[class_name]
        except Exception as e:
            print(f"Error loading label {label_path}: {e}")
            main_category = 0  # Default to organic if there's an error
            
        return image, main_category

class FolderWasteDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform
        
        print(f"\n📁 Loading Folder Dataset from: {self.root_dir}")
        
        # Get all images and their categories
        self.samples = []
        for category in tqdm(os.listdir(self.root_dir), desc="Scanning folders"):
            category_path = self.root_dir / category
            if category_path.is_dir() and category in FOLDER_CATEGORY_MAP:
                image_files = [f for f in os.listdir(category_path) 
                             if f.endswith(('.jpg', '.jpeg', '.png'))]
                for img_file in image_files:
                    self.samples.append((
                        category_path / img_file,
                        FOLDER_CATEGORY_MAP[category]
                    ))
                
                # Clear GPU cache periodically
                if len(self.samples) % 1000 == 0:
                    torch.cuda.empty_cache()
        
        print(f"✓ Found {len(self.samples):,} images")
        
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, category = self.samples[idx]
        
        try:
            image = Image.open(img_path).convert('RGB')
            if self.transform:
                image = self.transform(image)
            return image, category
        except Exception as e:
            print(f"Error loading image {img_path}: {e}")
            return torch.zeros((3, IMAGE_SIZE, IMAGE_SIZE)), 0

# Load YAML data
yaml_path = os.path.join(YOLO_PATH, 'data.yaml')
with open(yaml_path, 'r') as f:
    yaml_data = yaml.safe_load(f)
    class_names = yaml_data['names']

print("Found classes:", class_names)

Found classes: ['BIODEGRADABLE', 'CARDBOARD', 'GLASS', 'METAL', 'PAPER', 'PLASTIC']


## Data Preparation and Loading

In [ ]:
def prepare_data():
    print(f"{'='*80}")
    print("🔄 Preparing Datasets")
    
    # Load YAML with error handling
    try:
        yaml_path = os.path.join(YOLO_PATH, 'data.yaml')
        with open(yaml_path, 'r') as f:
            yaml_data = yaml.safe_load(f)
            class_names = yaml_data['names']
    except Exception as e:
        print(f"Error loading YAML: {e}")
        raise

    # Create datasets with smaller batch processing
    print("\nInitializing Datasets:")
    
    yolo_dataset = YOLOWasteDataset(
        os.path.join(YOLO_PATH, 'train', 'images'),
        class_names,
        data_transforms
    )
    
    folder_dataset = FolderWasteDataset(
        FOLDER_PATH,
        data_transforms
    )
    
    # Efficient label collection
    print("\nCollecting labels (this might take a minute)...")
    yolo_labels = []
    folder_labels = []
    
    # Process in smaller chunks
    chunk_size = 100
    for i in range(0, len(yolo_dataset), chunk_size):
        chunk = range(i, min(i + chunk_size, len(yolo_dataset)))
        yolo_labels.extend([yolo_dataset[j][1] for j in chunk])
        if i % 1000 == 0:
            print(f"Processed {i}/{len(yolo_dataset)} YOLO images")
            torch.cuda.empty_cache()
    
    for i in range(0, len(folder_dataset), chunk_size):
        chunk = range(i, min(i + chunk_size, len(folder_dataset)))
        folder_labels.extend([folder_dataset[j][1] for j in chunk])
        if i % 1000 == 0:
            print(f"Processed {i}/{len(folder_dataset)} folder images")
            torch.cuda.empty_cache()
    
    all_labels = yolo_labels + folder_labels
    class_counts = np.bincount(all_labels)
    
    print("\n📊 Class Distribution:")
    total = len(all_labels)
    for i, count in enumerate(class_counts):
        percentage = (count / total) * 100
        print(f"{CLASS_NAMES[i]:8s}: {count:5d} ({percentage:5.1f}%)")
    
    # Create balanced sampler
    print("\nCreating sampler...")
    class_weights = [total/count for count in class_counts]
    sample_weights = [class_weights[label] for label in all_labels]
    sampler = WeightedRandomSampler(sample_weights, len(sample_weights))
    
    # Combine datasets
    print("\nCombining datasets...")
    combined_dataset = ConcatDataset([yolo_dataset, folder_dataset])
    
    # Split train/val
    train_size = int(0.8 * len(combined_dataset))
    val_size = len(combined_dataset) - train_size
    
    train_dataset, val_dataset = torch.utils.data.random_split(
        combined_dataset, 
        [train_size, val_size],
        generator=torch.Generator().manual_seed(42)
    )
    
    # Create dataloaders with reduced workers and memory usage
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        sampler=sampler,
        num_workers=2,  # Reduced from 4
        pin_memory=True,
        prefetch_factor=2,
        persistent_workers=False,  # Changed to False
        drop_last=True
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,  # Reduced from 4
        pin_memory=True,
        persistent_workers=False  # Changed to False
    )
    
    print(f"\n✓ Final splits:")
    print(f"Training:   {train_size:,} images")
    print(f"Validation: {val_size:,} images")
    
    return train_loader, val_loader

# Prepare data with error handling
try:
    print("\nStarting data preparation...")
    train_loader, val_loader = prepare_data()
    
    print("\nTesting batch loading...")
    start_time = time.time()
    test_batch = next(iter(train_loader))
    load_time = time.time() - start_time
    
    print(f"Batch loaded in {load_time:.2f} seconds")
    print(f"Batch shapes - Images: {test_batch[0].shape}, Labels: {test_batch[1].shape}")
    print(f"Labels in batch: {test_batch[1].tolist()}")
    print(f"\nGPU Memory Status:")
    print_gpu_memory()
    
except Exception as e:
    print(f"\nError in data preparation: {str(e)}")
    raise


Starting data preparation...
🔄 Preparing Datasets

Initializing Datasets:

📁 Loading YOLO Dataset from: C:\Users\divya\OneDrive\Desktop\ECS\GARBAGE_YOLO\train\images


Validating files:   0%|          | 0/7324 [00:00<?, ?it/s]

✓ Found 7,324 valid images

📁 Loading Folder Dataset from: C:\Users\divya\OneDrive\Desktop\ECS\garbage_classification


Scanning folders:   0%|          | 0/12 [00:00<?, ?it/s]

✓ Found 15,515 images

Processed 0/7324 YOLO images
Processed 1000/7324 YOLO images
Processed 2000/7324 YOLO images
Processed 3000/7324 YOLO images
Processed 4000/7324 YOLO images
Processed 5000/7324 YOLO images
Processed 6000/7324 YOLO images
Processed 7000/7324 YOLO images
Processed 0/15515 folder images
Processed 1000/15515 folder images
Processed 2000/15515 folder images
Processed 3000/15515 folder images
Processed 4000/15515 folder images
Processed 5000/15515 folder images
Processed 6000/15515 folder images
Processed 7000/15515 folder images
Processed 8000/15515 folder images
Processed 9000/15515 folder images
Processed 10000/15515 folder images
Processed 11000/15515 folder images
Processed 12000/15515 folder images
Processed 13000/15515 folder images
Processed 14000/15515 folder images
Processed 15000/15515 folder images

📊 Class Distribution:
organic : 14707 ( 64.4%)
plastic :  1681 (  7.4%)
metal   :  2588 ( 11.3%)
glass   :  3863 ( 16.9%)

Creating sampler...

Combining datase

## Model Creation and Training

In [ ]:
def create_model():
    print("Creating MobileNetV2 model...")
    model = models.mobilenet_v2(pretrained=True)
    model.classifier[1] = torch.nn.Linear(model.classifier[1].in_features, NUM_CLASSES)
    
    # Move to GPU and optimize
    model = model.to(DEVICE)
    model = model.to(memory_format=torch.channels_last)
    
    return model

def train_epoch(model, loader, criterion, optimizer, epoch):
    losses = AverageMeter()
    accuracies = AverageMeter()
    
    model.train()
    stream = tqdm(loader, desc=f'Epoch {epoch+1} Training')
    
    for i, (inputs, targets) in enumerate(stream):
        # Move to GPU
        inputs = inputs.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)
        
        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        
        # Backward pass
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        
        # Calculate accuracy
        with torch.no_grad():
            _, preds = torch.max(outputs, 1)
            accuracy = torch.sum(preds == targets).float() / targets.size(0)
        
        # Update metrics
        losses.update(loss.item(), inputs.size(0))
        accuracies.update(accuracy.item(), inputs.size(0))
        
        # Update progress bar
        stream.set_postfix({
            'Loss': f'{losses.avg:.4f}',
            'Acc': f'{accuracies.avg:.4f}',
            'GPU Mem': f'{torch.cuda.memory_allocated()/1e9:.1f}GB'
        })
        
        # Clear cache periodically
        if i % 50 == 0:
            torch.cuda.empty_cache()
    
    return losses.avg, accuracies.avg

def validate(model, loader, criterion):
    losses = AverageMeter()
    accuracies = AverageMeter()
    
    model.eval()
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        stream = tqdm(loader, desc='Validation')
        for inputs, targets in stream:
            inputs = inputs.to(DEVICE, non_blocking=True)
            targets = targets.to(DEVICE, non_blocking=True)
            
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            
            _, preds = torch.max(outputs, 1)
            accuracy = torch.sum(preds == targets).float() / targets.size(0)
            
            losses.update(loss.item(), inputs.size(0))
            accuracies.update(accuracy.item(), inputs.size(0))
            
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(targets.cpu().numpy())
            
            stream.set_postfix({
                'Loss': f'{losses.avg:.4f}',
                'Acc': f'{accuracies.avg:.4f}'
            })
    
    return losses.avg, accuracies.avg, all_preds, all_targets

# Create model and optimizer
model = create_model()
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.1, patience=3)

# Training settings
NUM_EPOCHS = 20
best_accuracy = 0
training_history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': []
}

print("🚀 Starting training...")
start_time = time.time()

try:
    for epoch in range(NUM_EPOCHS):
        print(f"\n{'='*80}")
        print(f"Epoch {epoch+1}/{NUM_EPOCHS}")
        
        # Training
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, epoch)
        training_history['train_loss'].append(train_loss)
        training_history['train_acc'].append(train_acc)
        
        # Validation
        val_loss, val_acc, preds, targets = validate(model, val_loader, criterion)
        training_history['val_loss'].append(val_loss)
        training_history['val_acc'].append(val_acc)
        
        # Update learning rate
        scheduler.step(val_acc)
        
        # Save best model
        if val_acc > best_accuracy:
            best_accuracy = val_acc
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_accuracy': best_accuracy,
            }, 'best_model.pth')
            print(f"✓ Saved new best model with accuracy: {best_accuracy:.4f}")
        
        # Print epoch summary
        print(f"\nEpoch Summary:")
        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
        print_gpu_memory()
        
        # Plot progress every 5 epochs
        if (epoch + 1) % 5 == 0:
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
            
            ax1.plot(training_history['train_loss'], label='Train')
            ax1.plot(training_history['val_loss'], label='Validation')
            ax1.set_title('Loss vs Epoch')
            ax1.set_xlabel('Epoch')
            ax1.set_ylabel('Loss')
            ax1.legend()
            
            ax2.plot(training_history['train_acc'], label='Train')
            ax2.plot(training_history['val_acc'], label='Validation')
            ax2.set_title('Accuracy vs Epoch')
            ax2.set_xlabel('Epoch')
            ax2.set_ylabel('Accuracy')
            ax2.legend()
            
            plt.tight_layout()
            plt.show()
            
except KeyboardInterrupt:
    print("\n⚠️ Training interrupted by user")
except Exception as e:
    print(f"\n❌ Error during training: {str(e)}")
    raise
finally:
    torch.cuda.empty_cache()

time_elapsed = time.time() - start_time
print(f"\n✨ Training completed in {time_elapsed//3600:.0f}h {(time_elapsed%3600)//60:.0f}m {time_elapsed%60:.0f}s")
print(f"Best validation accuracy: {best_accuracy:.4f}")

## Model Evaluation and Export

In [ ]:
def evaluate_model():
    print("📊 Evaluating model performance...")
    
    # Load best model
    checkpoint = torch.load('best_model.pth')
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    # Get predictions
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for inputs, targets in tqdm(val_loader, desc="Getting predictions"):
            inputs = inputs.to(DEVICE)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(targets.numpy())
    
    # Calculate metrics
    print("\n📈 Classification Report:")
    print(classification_report(all_targets, all_preds, target_names=CLASS_NAMES))
    
    # Plot confusion matrix
    cm = confusion_matrix(all_targets, all_preds)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES,
                yticklabels=CLASS_NAMES)
    plt.title('Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()
    
    print("\n📊 Per-class Accuracy:")
    per_class_acc = cm.diagonal() / cm.sum(axis=1)
    for class_name, acc in zip(CLASS_NAMES, per_class_acc):
        print(f"{class_name:8s}: {acc:.4f}")
    
    return all_preds, all_targets

# Evaluate model
predictions, targets = evaluate_model()

In [ ]:
def export_to_onnx():
    print("💾 Exporting model to ONNX format...")
    
    # Load best model
    checkpoint = torch.load('best_model.pth')
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    # Create dummy input
    dummy_input = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE).to(DEVICE)
    
    try:
        # Export to ONNX
        torch.onnx.export(
            model,
            dummy_input,
            'waste_classifier.onnx',
            export_params=True,
            opset_version=11,
            do_constant_folding=True,
            input_names=['input'],
            output_names=['output'],
            dynamic_axes={
                'input': {0: 'batch_size'},
                'output': {0: 'batch_size'}
            }
        )
        
        print("\n✓ Model exported successfully")
        
        # Verify ONNX model
        import onnx
        onnx_model = onnx.load('waste_classifier.onnx')
        onnx.checker.check_model(onnx_model)
        print("✓ ONNX model verified")
        
        # Test inference
        import onnxruntime
        print("\n🧪 Testing inference...")
        
        session = onnxruntime.InferenceSession(
            'waste_classifier.onnx',
            providers=['CPUExecutionProvider']
        )
        
        # Run inference test
        input_name = session.get_inputs()[0].name
        test_input = np.random.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE).astype(np.float32)
        
        # Warmup
        _ = session.run(None, {input_name: test_input})
        
        # Test speed
        times = []
        for _ in range(10):
            start = time.time()
            _ = session.run(None, {input_name: test_input})
            times.append(time.time() - start)
        
        avg_time = sum(times) / len(times)
        print(f"Average inference time: {avg_time*1000:.2f} ms")
        print(f"FPS: {1/avg_time:.2f}")
        
        # Get model size
        model_size = os.path.getsize('waste_classifier.onnx') / (1024 * 1024)
        print(f"\nModel size: {model_size:.2f} MB")
        
    except Exception as e:
        print(f"\n❌ Error during export: {str(e)}")
        raise

# Export model
export_to_onnx()

print("\n✨ Complete! Model is ready for deployment on Raspberry Pi")
print("Use 'waste_classifier.onnx' with ONNX Runtime for inference")